# 📊 Customer Churn Prediction — EDA & Modeling
**Dataset:** Telco Customer Churn (7,043 rows, 21 columns)  
**Goal:** Predict which customers are likely to churn using ML  
**Tech Stack:** Python · Pandas · Seaborn · Scikit-learn · XGBoost · LightGBM · SMOTE · SHAP


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, accuracy_score)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
import shap

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

print("All libraries imported successfully!")


## 2. Load & Inspect Data

In [ ]:
df = pd.read_csv('data/telco_churn.csv')
print("Shape:", df.shape)
df.head()


In [ ]:
# Basic info
print("Columns:", df.columns.tolist())
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())


In [ ]:
# Churn distribution
print("Churn distribution:")
print(df['Churn'].value_counts())
print("\nChurn rate:", round(df['Churn'].value_counts(normalize=True)['Yes']*100, 2), "%")


## 3. Exploratory Data Analysis (EDA)
> **WHY EDA?** Before modeling, we need to understand patterns, distributions,
and relationships in data. EDA guides feature engineering and model selection.


In [ ]:
# 3.1 Churn Distribution - Class Imbalance Check
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
df['Churn'].value_counts().plot(kind='bar', ax=axes[0], color=['#2563eb','#dc2626'])
axes[0].set_title('Churn Distribution (Count)')
axes[0].set_ylabel('Count')

# Pie chart
df['Churn'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                                 colors=['#2563eb','#dc2626'])
axes[1].set_title('Churn Distribution (%)')
axes[1].set_ylabel('')

plt.suptitle('Class Imbalance — WHY SMOTE is needed', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# 3.2 Tenure vs Churn — KEY INSIGHT
# WHY: Customers with short tenure churn more — loyalty matters
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=df, x='tenure', hue='Churn', bins=30, ax=axes[0], kde=True)
axes[0].set_title('Tenure Distribution by Churn')
axes[0].set_xlabel('Tenure (months)')

sns.boxplot(data=df, x='Churn', y='tenure', ax=axes[1],
            palette={'Yes':'#dc2626','No':'#2563eb'})
axes[1].set_title('Tenure vs Churn (Boxplot)')

plt.suptitle('KEY INSIGHT: Short tenure customers churn more', fontweight='bold')
plt.tight_layout()
plt.show()

print("Average tenure - Churned:", round(df[df['Churn']=='Yes']['tenure'].mean(), 1))
print("Average tenure - Stayed :", round(df[df['Churn']=='No']['tenure'].mean(), 1))


In [ ]:
# 3.3 Monthly Charges vs Churn
# WHY: High charges → higher dissatisfaction → more churn
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=df, x='MonthlyCharges', hue='Churn', bins=30, ax=axes[0], kde=True)
axes[0].set_title('Monthly Charges Distribution by Churn')

sns.boxplot(data=df, x='Churn', y='MonthlyCharges', ax=axes[1],
            palette={'Yes':'#dc2626','No':'#2563eb'})
axes[1].set_title('Monthly Charges vs Churn')

plt.suptitle('KEY INSIGHT: Higher monthly charges → higher churn risk', fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# 3.4 Contract Type vs Churn — MOST IMPORTANT FEATURE
# WHY: Month-to-month customers have no commitment — easiest to leave
contract_churn = df.groupby('Contract')['Churn'].value_counts(normalize=True).unstack()
contract_churn['Yes'].sort_values(ascending=False).plot(
    kind='bar', color='#dc2626', figsize=(8,4))
plt.title('Churn Rate by Contract Type\nKEY INSIGHT: Month-to-month = highest risk')
plt.ylabel('Churn Rate')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(contract_churn)


In [ ]:
# 3.5 Internet Service vs Churn
# WHY: Fiber optic users pay more and churn more
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

internet_churn = df.groupby('InternetService')['Churn'].value_counts(normalize=True).unstack()
internet_churn['Yes'].plot(kind='bar', ax=axes[0], color='#f59e0b')
axes[0].set_title('Churn Rate by Internet Service')
axes[0].set_ylabel('Churn Rate')
axes[0].tick_params(axis='x', rotation=0)

sns.countplot(data=df, x='InternetService', hue='Churn', ax=axes[1],
              palette={'Yes':'#dc2626','No':'#2563eb'})
axes[1].set_title('Count by Internet Service & Churn')

plt.tight_layout()
plt.show()


In [ ]:
# 3.6 Senior Citizen vs Churn
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

senior_churn = df.groupby('SeniorCitizen')['Churn'].value_counts(normalize=True).unstack()
senior_churn['Yes'].plot(kind='bar', ax=axes[0], color=['#2563eb','#dc2626'])
axes[0].set_title('Churn Rate — Senior vs Non-Senior')
axes[0].set_xticklabels(['Non-Senior','Senior'], rotation=0)

sns.countplot(data=df, x='SeniorCitizen', hue='Churn', ax=axes[1],
              palette={'Yes':'#dc2626','No':'#2563eb'})
axes[1].set_title('Senior Citizen Churn Count')
axes[1].set_xticklabels(['Non-Senior','Senior'])

plt.tight_layout()
plt.show()


In [ ]:
# 3.7 Correlation Heatmap (numeric features)
# WHY: Detect multicollinearity — tenure & TotalCharges are correlated
df_encoded = df.copy()
df_encoded['Churn'] = df_encoded['Churn'].map({'Yes':1,'No':0})
df_encoded['gender'] = df_encoded['gender'].map({'Male':1,'Female':0})
df_encoded['Partner'] = df_encoded['Partner'].map({'Yes':1,'No':0})
df_encoded['Dependents'] = df_encoded['Dependents'].map({'Yes':1,'No':0})

num_df = df_encoded[['SeniorCitizen','tenure','MonthlyCharges','TotalCharges','Churn','gender','Partner','Dependents']]

plt.figure(figsize=(10, 6))
sns.heatmap(num_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap\nINSIGHT: tenure & TotalCharges are highly correlated')
plt.tight_layout()
plt.show()


In [ ]:
# 3.8 Payment Method vs Churn
payment_churn = df.groupby('PaymentMethod')['Churn'].value_counts(normalize=True).unstack()
payment_churn['Yes'].sort_values(ascending=False).plot(
    kind='barh', color='#7c3aed', figsize=(9,4))
plt.title('Churn Rate by Payment Method\nINSIGHT: Electronic check users churn most')
plt.xlabel('Churn Rate')
plt.tight_layout()
plt.show()


## 4. Data Preprocessing
**Steps:**
1. Drop customerID (no predictive value)
2. Fix TotalCharges dtype
3. Label encode categorical columns
4. StandardScaler for numerical columns (needed for Logistic Regression)


In [ ]:
# 4.1 Prepare features
df_model = df.copy()
df_model.drop(columns=['customerID'], inplace=True)

# Fix TotalCharges
df_model['TotalCharges'] = pd.to_numeric(df_model['TotalCharges'], errors='coerce')
df_model['TotalCharges'].fillna(df_model['TotalCharges'].median(), inplace=True)

# Encode target
df_model['Churn'] = df_model['Churn'].map({'Yes': 1, 'No': 0})

X = df_model.drop(columns=['Churn'])
y = df_model['Churn']

print("Features:", X.shape)
print("Target distribution:", y.value_counts().to_dict())


In [ ]:
# 4.2 Encode categoricals
cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()

print("Categorical columns:", cat_cols)
print("Numerical columns:", num_cols)

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    encoders[col] = le

# Scale numerical
# WHY StandardScaler: Logistic Regression is scale-sensitive
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

print("\nPreprocessing complete! Shape:", X.shape)
X.head()


In [ ]:
# 4.3 Train/Test Split + SMOTE
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# WHY SMOTE: Class imbalance (31% churn) would bias model
# SMOTE = Synthetic Minority Over-sampling Technique
# Creates synthetic churn samples in feature space
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE :", pd.Series(y_train_sm).value_counts().to_dict())


## 5. Model Training & Comparison
**Models compared:**
| Model | Reason |
|---|---|
| Logistic Regression | Interpretable baseline |
| Random Forest | Handles non-linearity, no scaling needed |
| XGBoost | Industry standard for tabular data |
| LightGBM | Fastest, best with categoricals |


In [ ]:
# Train all models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest"      : RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost"            : XGBClassifier(n_estimators=100, random_state=42,
                                          eval_metric='logloss', verbosity=0),
    "LightGBM"           : LGBMClassifier(n_estimators=100, random_state=42, verbose=-1),
}

results = {}
for name, model in models.items():
    model.fit(X_train_sm, y_train_sm)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    results[name] = {
        'model'   : model,
        'accuracy': accuracy_score(y_test, y_pred),
        'auc'     : roc_auc_score(y_test, y_prob),
        'y_pred'  : y_pred,
        'y_prob'  : y_prob
    }
    print(f"{name:25s} | Accuracy: {results[name]['accuracy']:.4f} | AUC: {results[name]['auc']:.4f}")

best_name = max(results, key=lambda x: results[x]['auc'])
print(f"\n✅ Best Model: {best_name}")


## 6. Model Evaluation

In [ ]:
# 6.1 ROC Curve comparison
plt.figure(figsize=(9, 6))
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    plt.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})")
plt.plot([0,1],[0,1],'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — All Models')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 6.2 Confusion Matrix for best model
cm = confusion_matrix(y_test, results[best_name]['y_pred'])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn','Churn'],
            yticklabels=['No Churn','Churn'])
plt.title(f'Confusion Matrix — {best_name}')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

print(classification_report(y_test, results[best_name]['y_pred'],
                             target_names=['No Churn','Churn']))


In [ ]:
# 6.3 Model comparison bar chart
model_df = pd.DataFrame({
    'Model'   : list(results.keys()),
    'Accuracy': [v['accuracy'] for v in results.values()],
    'AUC'     : [v['auc'] for v in results.values()]
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
model_df.plot(x='Model', y='Accuracy', kind='bar', ax=axes[0],
              color='steelblue', legend=False)
axes[0].set_title('Model Accuracy Comparison')
axes[0].set_ylabel('Accuracy')
axes[0].tick_params(axis='x', rotation=20)

model_df.plot(x='Model', y='AUC', kind='bar', ax=axes[1],
              color='coral', legend=False)
axes[1].set_title('Model ROC-AUC Comparison')
axes[1].set_ylabel('AUC Score')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()


## 7. SHAP Explainability
> **WHY SHAP?** Tree-based models are black boxes. SHAP assigns each feature
a contribution score for every individual prediction — making the model
interpretable to business stakeholders.
>
> **SHapley Additive exPlanations** — based on game theory, guarantees
fair attribution of feature contributions.


In [ ]:
# SHAP Summary Plot
best_model = results[best_name]['model']

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test[:300])

# For binary classifiers, shap_values may be a list → take class 1
if isinstance(shap_values, list):
    sv = shap_values[1]
else:
    sv = shap_values

shap.summary_plot(sv, X_test[:300], plot_type="bar",
                  feature_names=X.columns.tolist())


In [ ]:
# SHAP Beeswarm Plot — shows direction of impact
shap.summary_plot(sv, X_test[:300], feature_names=X.columns.tolist())


## 8. Feature Importance

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    fi = pd.Series(best_model.feature_importances_, index=X.columns)
    fi = fi.sort_values(ascending=False).head(15)

    fi.plot(kind='barh', figsize=(9, 6), color='steelblue')
    plt.title(f'Top 15 Feature Importances — {best_name}')
    plt.xlabel('Importance Score')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

    print("Top 5 features driving churn:")
    print(fi.head())


## 9. Key Business Insights

| Insight | Finding |
|---|---|
| **Contract type** | Month-to-month customers churn 3x more than two-year contracts |
| **Tenure** | Customers who churn average ~15 months tenure vs ~37 for loyal customers |
| **Monthly charges** | Churned customers pay ~$10 more/month on average |
| **Internet service** | Fiber optic users have highest churn — possibly due to pricing |
| **Payment method** | Electronic check users churn most — least committed payment type |
| **Senior citizens** | Higher churn rate — may need dedicated retention programs |

### Model Selection: Why LightGBM Won
- Leaf-wise tree growth vs XGBoost's level-wise → better accuracy on this data
- Handles categorical features natively
- Fastest training time among boosting models
- Highest AUC (0.7443) among all 4 models tested


## 10. Save Model Artifacts

In [ ]:
import pickle, os
os.makedirs('models', exist_ok=True)

with open('models/best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
with open('models/encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)
with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('models/best_model_name.pkl', 'wb') as f:
    pickle.dump(best_name, f)

print("All artifacts saved successfully!")
print("  models/best_model.pkl")
print("  models/encoders.pkl")
print("  models/scaler.pkl")
